# Visual Emotion Recognition — Colab runner
Runtime → Change runtime type → **T4 GPU**.

By default outputs stay inside this Colab session (`/content/emotion_out`) and **Google Drive is not used**. Download the zip at the end before closing the tab, because Colab deletes its files when the session ends.

In [ ]:
!nvidia-smi -L

USE_DRIVE = False   # True = keep outputs in Google Drive (survives disconnects)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/Renad87/computer-vision.git
%cd /content/computer-vision
!git checkout feature/emotion-training
!pip install -q -r ml/emotion/requirements.txt

In [ ]:
import os
OUT = '/content/drive/MyDrive/poetry_vision_emotion' if USE_DRIVE else '/content/emotion_out'
os.makedirs(OUT, exist_ok=True)
DIRS = f'--features_dir {OUT}/features --artifacts_dir {OUT}/artifacts --results_dir {OUT}/results'
CACHE = f'--cache_dir {OUT}/cache'
print('Outputs ->', OUT)

## 1) Sample EmoSet + extract features
Faster: add `--per_class_train 600 --skip_resnet`.

In [ ]:
!python ml/emotion/prepare_features.py --features_dir {OUT}/features --results_dir {OUT}/results {CACHE}

## 2) Train: zero-shot → feature extraction → fine-tuning

In [ ]:
!python ml/emotion/train.py {DIRS} {CACHE} --finetune

## 3) Independent test evaluation

In [ ]:
!python ml/emotion/evaluate.py {DIRS} {CACHE}
from IPython.display import Markdown, Image, display
display(Markdown(open(f'{OUT}/results/RESULTS.md').read()))
for f in ['model_comparison.png', 'confusion_clip_finetuned.png', 'ambiguous_examples.png', 'attributes_by_emotion.png']:
    p = f'{OUT}/results/{f}'
    if os.path.exists(p): display(Image(p))

## 4) Try the integration function

In [ ]:
import sys, json
sys.path.insert(0, '/content/computer-vision')
from google.colab import files
up = files.upload()
from ml.emotion.inference import EmotionAnalyzer
an = EmotionAnalyzer(artifacts_dir=f'{OUT}/artifacts')
for name, data in up.items():
    print(json.dumps(an.analyze(data), ensure_ascii=False, indent=2))

## 5) Download results
Downloads `results/` and `artifacts/` (no images, no features).

In [ ]:
from google.colab import files
!rm -f /content/emotion_results.zip && cd {OUT} && zip -qr /content/emotion_results.zip results artifacts
files.download('/content/emotion_results.zip')